## Baseline Hardcoded Parameter Study

This notebook performs a **controlled exploration of the baseline parameter space** to justify the use of Bayesian optimization (Optuna) and to identify stable representational regimes.

The analysis focuses on the interaction between:
- TF-IDF pruning (`max_df`)
- word and character n-gram ranges
- regularization strength (`C`) of the Logistic Regression

All experiments are conducted on the **development split only**, with Macro F1 as the evaluation metric.

### Key Results

A clear and stable performance plateau emerges for the following configuration:
- `word_ng = 2`
- `char_ng = 5`
- `C ∈ [0.6, 1.0]`
- `max_df ∈ [0.80, 0.92]`

Top configurations consistently cluster in this region:

| C | max_df | word_ng | char_ng | Macro F1 |
|---|--------|---------|---------|----------|
| 1.0 | 0.85 | 2 | 5 | 0.7193 |
| 1.0 | 0.90 | 2 | 5 | 0.7191 |
| 0.6 | 0.90 | 2 | 5 | 0.7190 |
| 0.6 | 0.85 | 2 | 5 | 0.7188 |

Performance gains are **driven by representational choices** (n-gram structure), while fine-grained variations of `C` and `max_df` yield negligible differences.

### Conclusion

This study shows that:
- the baseline reaches a **robust local optimum** once the correct TF-IDF structure is selected,
- further manual tuning provides diminishing returns,
- Bayesian optimization is justified only within this constrained and empirically validated region.

The selected configuration is therefore **hardcoded** in the baseline and used as the reference point for subsequent Optuna-based refinement.


In [3]:
#Libraries

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


In [4]:
#Parameters

C_VALUES        = [0.1, 0.3, 0.6, 1.0, 1.5, 2.0]
MAX_DF_VALUES   = [0.80, 0.85, 0.88, 0.90, 0.92]
WORD_NG_VALUES  = [1, 2]
CHAR_NG_VALUES  = [3, 5]

MIN_DF = 2

NUM_COLS = [
	"n_tokens",
	"title_len",
	"article_len",
	"title_ratio"
]


In [5]:
#Load data 

In [ ]:
DEV_PATH = "../../data/processed/development_processed.csv"
df = pd.read_csv(DEV_PATH)

# sicurezza minima
for col in ["source", "text"]:
	df[col] = df[col].fillna("").astype(str)

X = df[["source", "text"] + NUM_COLS]
y = df["label"]

X_tr, X_te, y_tr, y_te = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

In [7]:
#MODELS 

results = []

for C in C_VALUES:
	for max_df in MAX_DF_VALUES:
		for word_ng in WORD_NG_VALUES:
			for char_ng in CHAR_NG_VALUES:

				model = Pipeline([
					("pre", ColumnTransformer(
						transformers=[
							("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

							("w_tfidf", TfidfVectorizer(
								analyzer="word",
								ngram_range=(1, word_ng),
								min_df=MIN_DF,
								max_df=max_df,
								sublinear_tf=True,
								max_features=250_000
							), "text"),

							("c_tfidf", TfidfVectorizer(
								analyzer="char_wb",
								ngram_range=(3, char_ng),
								min_df=MIN_DF,
								max_df=max_df,
								sublinear_tf=True,
								max_features=300_000
							), "text"),

							("num", StandardScaler(), NUM_COLS),
						],
						remainder="drop",
						n_jobs=-1
					)),
					("clf", LogisticRegression(
						C=C,
						class_weight="balanced",
						max_iter=2000,
						n_jobs=-1
					))
				])

				model.fit(X_tr, y_tr)
				pred = model.predict(X_te)

				f1 = f1_score(y_te, pred, average="macro")

				results.append({
					"C": C,
					"max_df": max_df,
					"word_ng": word_ng,
					"char_ng": char_ng,
					"macro_f1": f1
				})

				print(
					f"C={C}, max_df={max_df}, "
					f"w_ng={word_ng}, c_ng={char_ng} -> F1={f1:.4f}"
				)


C=0.1, max_df=0.8, w_ng=1, c_ng=3 -> F1=0.6958
C=0.1, max_df=0.8, w_ng=1, c_ng=5 -> F1=0.6995
C=0.1, max_df=0.8, w_ng=2, c_ng=3 -> F1=0.6931
C=0.1, max_df=0.8, w_ng=2, c_ng=5 -> F1=0.6992
C=0.1, max_df=0.85, w_ng=1, c_ng=3 -> F1=0.6959
C=0.1, max_df=0.85, w_ng=1, c_ng=5 -> F1=0.6992
C=0.1, max_df=0.85, w_ng=2, c_ng=3 -> F1=0.6926
C=0.1, max_df=0.85, w_ng=2, c_ng=5 -> F1=0.6993
C=0.1, max_df=0.88, w_ng=1, c_ng=3 -> F1=0.6957
C=0.1, max_df=0.88, w_ng=1, c_ng=5 -> F1=0.6990
C=0.1, max_df=0.88, w_ng=2, c_ng=3 -> F1=0.6930
C=0.1, max_df=0.88, w_ng=2, c_ng=5 -> F1=0.6995
C=0.1, max_df=0.9, w_ng=1, c_ng=3 -> F1=0.6957
C=0.1, max_df=0.9, w_ng=1, c_ng=5 -> F1=0.6990
C=0.1, max_df=0.9, w_ng=2, c_ng=3 -> F1=0.6930
C=0.1, max_df=0.9, w_ng=2, c_ng=5 -> F1=0.6995
C=0.1, max_df=0.92, w_ng=1, c_ng=3 -> F1=0.6957
C=0.1, max_df=0.92, w_ng=1, c_ng=5 -> F1=0.6990
C=0.1, max_df=0.92, w_ng=2, c_ng=3 -> F1=0.6930
C=0.1, max_df=0.92, w_ng=2, c_ng=5 -> F1=0.6995
C=0.3, max_df=0.8, w_ng=1, c_ng=3 -> F1=0.7094
C

In [8]:
results_df = pd.DataFrame(results)
results_df.to_csv("/kaggle/working/baseline_grid_results.csv", index=False)

print("\nTop configurations:")
print(results_df.sort_values("macro_f1", ascending=False).head(10))


Top configurations:
      C  max_df  word_ng  char_ng  macro_f1
67  1.0    0.85        2        5  0.719268
75  1.0    0.90        2        5  0.719063
71  1.0    0.88        2        5  0.719063
79  1.0    0.92        2        5  0.719063
55  0.6    0.90        2        5  0.719016
51  0.6    0.88        2        5  0.719016
59  0.6    0.92        2        5  0.719016
43  0.6    0.80        2        5  0.718922
47  0.6    0.85        2        5  0.718849
63  1.0    0.80        2        5  0.718802
